In [ ]:
#w2 ct4
#ส่วน Import
import json
import os
import csv
import random
import time
from paho.mqtt import client as mqtt_client

#การตังค่าไฟล์และโฟลเดอร์
# TODO: UPDATE FOLDER PATH and OUTPUT FILE-NAME IN HERE
SAVE_FOLDER = "Data"
CSV_FILENAME = "received_plant_data.csv"
OUTPUT_CSV = os.path.join(SAVE_FOLDER, CSV_FILENAME)                        #set output ว่าให้ออกตรงไหน

#เก็บสถานะการทำงาน
STATE = {
    "writer_initialized": False,                                            #เช็คว่าโปรแกรมได้เตรียมการเขียนไฟล์ CSV (สร้าง Header) ไปแล้วหรือยัง
    "fieldnames": None,                                                     #เก็บรายชื่อหัวคอลัมน์ที่ดึงมาจากข้อมูล JSON ชุดแรก
    "first_message_received": False,                                        #เป็น Flag ว่าได้รับข้อความแรกหรือยังf
    "last_msg_time": None,                                                  #เก็บเวลาล่าสุดที่ได้รับข้อมูลจาก MQTT
}

#การตังค่า MQTT
# NOTE: MQTT CONFIGURATION
MQTT_CONFIG = {
    "BROKER": "broker.emqx.io",
    "PORT": 1883,
    "TOPIC": "idt/plant/env",                                               #TOPICเปลี่ยนชื่อidt ตามต้องการ
    "CLIENT_ID": f"ALPHA-I-{random.randint(0,100)}",
    "KEEPALIVE": 120,
}

#การเชือมต่อ MQTT
def connect_mqtt():
    client = mqtt_client.Client(mqtt_client.CallbackAPIVersion.VERSION1, MQTT_CONFIG["CLIENT_ID"])      #สร้างตัวแปร Clientเพื่อไปคุยกับ Server
    client.on_connect = on_connect                                                                      #ถ้าเชื่อมต่อสำเร็จแล้ว ให้ไปรันฟังก์ชัน
    client.on_message = on_message                                                                      #ถ้ามีข้อความถูกส่งกลับมา ให้ไปรันฟังก์ชัน
    client.connect(MQTT_CONFIG["BROKER"], MQTT_CONFIG["PORT"], MQTT_CONFIG["KEEPALIVE"])                #เชื่อมต่อBROKER,PORT,KEEPALIVE ที่setไว้
    return client

#เมือเชือมต่อสําเร็จ
def on_connect(client, userdata, flags, rc):
    if rc == 0:                                                             #ถ้าค่า rc เท่ากับ 0 หมายความว่า "การเชื่อมต่อสำเร็จ"
        print("Connected to MQTT broker.")                                  #แสดงข้อความเพื่อให้รู้ว่าเชื่อมต่อกับ Server ได้แล้ว
        client.subscribe(MQTT_CONFIG["TOPIC"])                              #เมื่อต่อติดแล้ว จะสั่งให้ Client ไป "ติดตาม" (Subscribe) หัวข้อ (Topic) ที่เรากำหนดไว้ในคอนฟิก เพื่อรอรับข้อมูลที่ถูกส่งเข้ามา
        print(f"Subscribed to topic: {MQTT_CONFIG['TOPIC']}")               #ยืนยันว่าตอนนี้เรากำลังติดตามหัวข้ออะไรอยู่
    else:
        print(f"Failed to connect, return code {rc}")                       #แจ้งเตือนว่าเชื่อมต่อไม่สำเร็จ พร้อมแสดงรหัสข้อผิดพลาด

#บันทึกข้อมูลลง CSV
def save_data_csv(data: dict):
    os.makedirs(SAVE_FOLDER, exist_ok=True)                                 #สร้างโฟลเดอร์สำหรับเก็บไฟล์

    if not STATE["writer_initialized"]:
        STATE["fieldnames"] = list(data.keys())                             #ดึงชื่อ Key ทั้งหมดจากข้อมูลชุดแรกมาตั้งเป็นชื่อหัวคอลัมน์ของ CSV
        file_exists = os.path.isfile(OUTPUT_CSV)                            #เช็คดูว่าในเครื่องมีไฟล์นี้อยู่ก่อนแล้วหรือไม่

        with open(OUTPUT_CSV, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=STATE["fieldnames"])
            if not file_exists:
                writer.writeheader()
            writer.writerow(data)

        STATE["writer_initialized"] = True                                  #เปลี่ยนสถานะเป็น True เพือให้ครั้งต่อไปไม่ต้องเช็คหัวตาราง แล้วเขียนข้อมูลลงไปได้เลย
        return

    with open(OUTPUT_CSV, "a", newline="", encoding="utf-8") as f:          #เปิดไฟล์ในโหมด "a" การเขียนต่อท้ายไฟล์เดิม ไม่ใช่การเขียนทับ
        writer = csv.DictWriter(f, fieldnames=STATE["fieldnames"])          #ใช้ตัวช่วยเขียนไฟล์ CSV แบบดึงข้อมูลจาก Dictionary โดยตรง
        writer.writerow(data)                                               #นำข้อมูลในตัวแปร data ไปเขียนลงเป็นบรรทัดใหม่ในไฟล์ CSV

#เมือได้รับข้อความ
def on_message(client, userdata, msg):                                  #ฟังก์ชัน Callback ที่จะถูกเรียกเมื่อมีข้อความเข้า
    STATE["first_message_received"] = True                              #บันทึกสถานะลงในตัวแปร STATE
    STATE["last_msg_time"] = time.time()                                #บันทึกเวลาปัจจุบันที่ได้รับข้อความล่าสุด

    try:
        payload_str = msg.payload.decode("utf-8")                       #แปลงข้อมูลที่ได้รับให้กลายเป็น ข้อความ (String)
        data = json.loads(payload_str)                                  #แปลงข้อความ String ที่อยู่ในรูปแบบ JSON ให้กลายเป็น Dictionary เพื่อเรียกใช้งานข้อมูลตาม Key ต่างๆ ได้ง่าย
        save_data_csv(data)                                             #นำข้อมูลที่แปลงแล้วไปเขียนลงไฟล์ CSV
        print(f"Saved row to {OUTPUT_CSV}: {data}")                     #แสดงข้อความยืนยันในหน้าจอว่าบันทึกข้อมูล พร้อมโชว์ข้อมูล

    except Exception as e:                                              #ถ้ามีอะไรผิดพลาดในขั้นตอนด้านบน โปรแกรมจะมาที่นี่ทันที
        print(f"Error processing message: {e}")                         #พิมพ์ข้อผิดพลาดที่เกิดขึ้นออกมา

#เริมโปรแกรม
def subscriber(IDLE_TIMEOUT = 10):
    client = connect_mqtt()                                                                     #สั่งให้ MQTT ทำงานใน เพื่อคอยรับข้อความตลอดเวลาโดยไม่ทำให้โปรแกรมหลักหยุดชะงัก
    try:
    print("Starting MQTT Subscribe. (Timeout Activate)")

    client.loop_start()
    try:
        while True:                                                                             #เฝ้าดูเงื่อนไข
            if STATE["first_message_received"]:                                                 #ถ้าเริ่มได้รับข้อมูลแล้วจะเริ่มคำนวณเวลาว่าง (Idle Time)
                idle = time.time() - STATE["last_msg_time"]                                     #การเอาเวลาปัจจุบัน ลบด้วยเวลาที่ได้รับข้อความล่าสุด
                if idle >= IDLE_TIMEOUT:                                                        #ถ้า idleเวลาที่เงียบหายไปนานเกินกว่าค่าที่กำหนดโปรแกรมจะสั่ง break เพื่อหยุดการทำงาน
                    print(
                        f"No Subscribe message for {IDLE_TIMEOUT}s -- stop subscribe process."
                    )
                    break
            time.sleep(0.5)

    except KeyboardInterrupt:                                                                   #ถ้ากด Ctrl + Cโปรแกรมจะหยุดทำงานอย่างปกติ แทนการปิดไปแบบดื้อๆ
        print("\nStopping subscriber manually...")

    finally:
        client.loop_stop()                                                                      #สั่งหยุดการทำงาน
        client.disconnect()                                                                     #ยกเลิกการเชื่อมต่อ
        print("Disconnected from MQTT broker.")


if __name__ == "__main__":
    subscriber()